# Docker HDDM 参数拟合工作流 — v2.0

**项目**: GP-SPE 实验设计优化 — Self-Matching Task DDM 参数提取

---

## v2.0 更新说明（相比 v1.0）

| 改进项 | 类别 | 说明 |
|--------|------|------|
| 遗漏试次处理 | **严重修复** | v1 将遗漏试次标记为 response=0 导致 HDDM 误将其当作错误反应；v2 改为在拟合时排除遗漏试次，仅对有效试次建模 |
| Rhat 收敛诊断 | **新增** | 使用 `gen_stats()` 输出 Rhat 值并检查所有参数 Rhat < 1.01 |
| 收敛可视化 | **新增** | 迹线图 (trace plot) + 后验密度图 (posterior density) |
| 后验预测检查 (PPC) | **新增** | 使用 `hddm.utils.post_pred_gen()` 生成后验预测，比较观测 vs 预测 RT 分布 |
| DIC 拟合优度 | **新增** | 报告每组 DIC 值 |
| 参数边界检查 | **新增** | 检查各参数 95% CI 是否触及边界 |
| SPE_v 误差条 | **修复** | v1 仅用 v_self 不确定度，v2 从后验样本直接计算 SPE_v 分布 |
| 被试水平参数 | **新增** | 提取并保存个体水平 DDM 参数 |
| MCMC 数据库保留 | **改进** | 不再删除 .db 文件，供后续诊断使用 |

---

## ⚠️ Docker 启动方式（重要）

在 CMD/PowerShell 中执行以下命令启动 Docker Jupyter：

```bash
docker pull hcp4715/hddm

docker run -it --rm --cpus=4 ^
  -v /d/GitHub_programe/GitHub/Guassion-Process-Experiment-Design:/home/jovyan/work ^
  -p 8888:8888 ^
  hcp4715/hddm ^
  jupyter notebook
```

> **注意**: 挂载路径是**项目根目录**（而非 Python_HDDM 子目录），这样数据直接写入正确的 `2_Data/` 和 `3_Figures/` 位置。

---

## 工作流步骤

| 步骤 | 环境 | 内容 |
|------|------|------|
| Step 1 | Docker（本文件） | 数据预处理 → HDDM 就绪 CSV（排除遗漏试次） |
| Step 2 | Docker（本文件） | HDDM 层级模型 MCMC 拟合 + 收敛诊断 + DIC |
| Step 3 | Docker（本文件） | 后验预测检查 (PPC) |
| Step 4 | Docker（本文件） | 参数提取（组水平 + 被试水平） |
| Step 5 | Docker（本文件） | 综合可视化（收敛图 + 参数图） |

---
## 环境检查

In [ ]:
# ============================================================
# 环境检查
# ============================================================
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
import pickle
import matplotlib.pyplot as plt
import re
import os
warnings.filterwarnings("ignore")

BASE_DIR = Path("/home/jovyan/work")
DATA_DIR = BASE_DIR / "2_Data" / "Real_Data" / "HDDM_Ready"
OUT_DIR  = BASE_DIR / "2_Data" / "Real_Data" / "HDDM_Traces"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR:  {BASE_DIR}  存在: {BASE_DIR.exists()}")
print(f"DATA_DIR:  {DATA_DIR}  存在: {DATA_DIR.exists()}")
print(f"OUT_DIR:   {OUT_DIR}")

try:
    import hddm
    print(f"HDDM 版本: {hddm.__version__}")
except ImportError:
    print("❌ HDDM 未安装！请在 Docker 容器 (hcp4715/hddm) 中运行此笔记本。")
    sys.exit(1)

csv_files = sorted(DATA_DIR.glob("hddm_data_group*.csv"))
if not csv_files:
    print(f"⚠️ 未找到 HDDM 就绪数据！Step 1 将自动预处理。")
    print(f"   预期路径: {DATA_DIR}")
else:
    print(f"✅ 发现 {len(csv_files)} 个待拟合数据文件")
    for f in csv_files:
        df = pd.read_csv(f)
        n_subj = df["subj_idx"].nunique()
        n_omis = (df.get("omission", 0) == 1).sum()
        print(f"   {f.name}: {n_subj}被试, {len(df)}试次, 遗漏{n_omis}")

---
## Step 1: 数据预处理（v2 改进）

从原始 CSV 中提取 Matching 试次，**排除遗漏试次**，输出仅含有效试次的 HDDM 就绪格式。

> ⚠️ **v2 核心改动**: v1 将遗漏试次的 response 设为 0 导致 HDDM 将其误解释为"错误反应"，v2 改为在拟合时完全排除遗漏试次。遗漏率作为独立行为指标保存。

> ⚠️ 如果 `2_Data/Real_Data/HDDM_Ready/` 下已有 `hddm_data_group*.csv` 文件，可以跳过此步骤直接进入 Step 2。

In [ ]:
# ============================================================
# Step 1: 数据预处理（v2: 排除遗漏试次）
# ============================================================

RAW_DIR = BASE_DIR / "2_Data" / "Real_Data" / "UnExtact" / "raw"
READY_DIR = BASE_DIR / "2_Data" / "Real_Data" / "HDDM_Ready"
READY_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Step 1: 数据预处理（v2: 排除遗漏试次）")
print("=" * 60)

raw_files = sorted(RAW_DIR.glob("EXP_data_group*.csv"))
print(f"\n发现 {len(raw_files)} 个原始数据文件")

dfs = []
for f in raw_files:
    df = pd.read_csv(f)
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print(f"合并总行数: {len(df_all)}")

print("\n各组被试分布:")
group_subjects = df_all.groupby("groupID")["subjectID"].nunique()
for gid in sorted(group_subjects.index):
    print(f"  Group {gid}: {group_subjects[gid]} 名被试")

# 过滤 formal 阶段
df_f = df_all[df_all["stage"] == "formal"].copy()
print(f"\n过滤 stage='formal' 后: {len(df_f)} 行")

# 仅保留 Matching 试次: (circle,self) 或 (square,stranger)
matching_mask = (
    ((df_f["Shape"] == "circle") & (df_f["Label"] == "self"))
    | ((df_f["Shape"] == "square") & (df_f["Label"] == "stranger"))
)
df_match = df_f[matching_mask].copy()
print(f"过滤 Matching 试次后: {len(df_match)} 行")

# 构建 HDDM 所需列
df_match["identity"] = df_match["Label"].map({"self": 1, "stranger": 0})
df_match["RT_num"] = pd.to_numeric(df_match["RT"], errors="coerce")
df_match["omission"] = df_match["RT_num"].isna().astype(int)
df_match["T_s"] = pd.to_numeric(df_match["T"], errors="coerce")
df_match["W_s"] = pd.to_numeric(df_match["W"], errors="coerce")

# response: 正确=1, 错误=0
df_match["response"] = np.where(df_match["Correct"] == 1, 1, 0)

print("\n各组 HDDM 就绪数据 (v2: 仅含有效试次):")

omission_summary = []  # 记录遗漏率行为指标

for gid in sorted(df_match["groupID"].unique()):
    gdf = df_match[df_match["groupID"] == gid].copy()

    # 验证组内 T 值一致性
    T_vals = (gdf["T_s"] * 1000).astype(int)
    unique_T = T_vals.unique()
    if len(unique_T) > 1:
        print(f"  ⚠️ Group {gid} T值不一致: {unique_T}")

    # 排除遗漏试次（v2 关键改动）
    n_total = len(gdf)
    n_omission = gdf["omission"].sum()
    gdf_valid = gdf[gdf["omission"] == 0].copy()

    # 被试编号映射
    subj_map = {s: i for i, s in enumerate(sorted(gdf_valid["subjectID"].unique()))}
    gdf_valid["subj_idx"] = gdf_valid["subjectID"].map(subj_map)

    # HDDM 所需列（v2: 不再包含 omission 列）
    hddm_cols = ["subj_idx", "RT_num", "response", "identity"]
    hddm_df = gdf_valid[hddm_cols].copy()
    hddm_df = hddm_df.rename(columns={"RT_num": "rt"})

    # 从实际数据获取实验参数
    P_val = int(gdf["P"].iloc[0])
    T_val = int(T_vals.mode().iloc[0] if len(unique_T) > 1 else unique_T[0])
    W_val = int((gdf["W_s"].iloc[0] * 1000))

    fn = f"hddm_data_group{gid}_P{P_val}_T{T_val}_W{W_val}.csv"
    out_path = READY_DIR / fn
    hddm_df.to_csv(out_path, index=False)

    n_subj = gdf_valid["subjectID"].nunique()
    n_trials = len(hddm_df)
    omission_rate = n_omission / n_total * 100
    acc = hddm_df["response"].mean()
    self_rt = hddm_df[hddm_df["identity"] == 1]["rt"].mean()
    stranger_rt = hddm_df[hddm_df["identity"] == 0]["rt"].mean()

    print(f"  Group {gid} (P={P_val}, T={T_val}ms, W={W_val}ms) → {fn}")
    print(f"    被试: {n_subj} | 有效试次: {n_trials}/{n_total}")
    print(f"    遗漏率: {omission_rate:.1f}% | 正确率: {acc:.3f}")
    print(f"    RT_Self: {self_rt:.3f}s | RT_Stranger: {stranger_rt:.3f}s")

    omission_summary.append({
        "group_id": gid, "P": P_val, "T_ms": T_val, "W_ms": W_val,
        "n_subjects": n_subj, "n_total": n_total, "n_valid": n_trials,
        "n_omissions": n_omission, "omission_rate": omission_rate,
        "accuracy": acc,
        "rt_self_mean": self_rt, "rt_stranger_mean": stranger_rt
    })

# 保存遗漏率行为指标
df_omission = pd.DataFrame(omission_summary)
df_omission.to_csv(OUT_DIR / "omission_behavioral_summary.csv", index=False)
print(f"\n✅ 遗漏率行为指标已保存到: {OUT_DIR / 'omission_behavioral_summary.csv'}")

# 更新 csv_files 变量供后续 Step 2 使用
csv_files = sorted(READY_DIR.glob("hddm_data_group*.csv"))
print(f"\n✅ HDDM 就绪数据已保存到: {READY_DIR}")
print(f"准备拟合 {len(csv_files)} 个文件（仅含有效试次）")

---
## Step 2: HDDM 层级模型 MCMC 拟合（v2 改进）

对每组实验条件独立拟合层级 DDM，使用 `depends_on={"v": "identity"}` 区分 Self/Stranger 漂移率。

**v2 新增**: 
- 拟合后计算 Rhat 值并检查收敛
- 计算 DIC 拟合优度
- 参数边界检查
- 保留 MCMC 数据库文件

**预估时间**: 每组约 2-5 分钟，全部 8 组约 15-40 分钟。

In [ ]:
# ============================================================
# Step 2: HDDM 层级模型 MCMC 拟合 (v2)
# ============================================================

DB_DIR = OUT_DIR / "mcmc_databases"
DB_DIR.mkdir(parents=True, exist_ok=True)

# 存储每组模型的引用（供后续 DIC 等使用）
all_models = {}
all_rhat_warnings = []  # 记录收敛警告
all_boundary_warnings = []  # 记录边界警告

for csv_path in csv_files:
    fname = csv_path.stem
    print(f"\n{'=' * 50}")
    print(f"拟合: {fname}")
    print(f"{'=' * 50}")

    df = pd.read_csv(csv_path)
    n_subj = df["subj_idx"].nunique()
    n_self = (df["identity"] == 1).sum()
    n_stranger = (df["identity"] == 0).sum()
    acc = df["response"].mean()

    print(f"  被试: {n_subj}, 有效试次: {len(df)}")
    print(f"  Self: {n_self}, Stranger: {n_stranger}")
    print(f"  正确率: {acc:.3f}")

    # 构建 HDDM 模型（v2: 不包含遗漏试次，无需 omission 列）
    model = hddm.HDDM(
        df,
        depends_on={"v": "identity"},
        include=["v", "a", "t", "z"],
        bias=False,
        p_outlier=0.05,
    )

    # MCMC 采样
    db_name = str(DB_DIR / f"traces_{fname}.db")
    print(f"  开始 MCMC 采样 (3000 draws, 500 burn)...")
    model.sample(3000, burn=500, dbname=db_name, db="pickle")
    print("  采样完成")

    # ---- v2 新增: 生成统计摘要（含 Rhat） ----
    stats = model.gen_stats()
    stats_path = OUT_DIR / f"{fname}_stats.csv"
    stats.to_csv(stats_path)
    print(f"  统计 (含 Rhat) -> {stats_path}")

    # 检查 Rhat 收敛
    rhat_cols = [c for c in stats.columns if "r_hat" in c.lower() or "Rhat" in c]
    if len(rhat_cols) == 0:
        # gen_stats 可能直接包含 Rhat 列名
        # 尝试查找 gearman_rubin 或类似的列
        if "Rhat" not in stats.columns:
            print("  ⚠️ gen_stats 未返回 Rhat 列，可能为单链采样")
            print("  → 将使用迹线图 (Step 4) 进行收敛定性检查")
    else:
        rhat_col = rhat_cols[0]
        rhat_vals = stats[rhat_col].dropna()
        # 仅检查核心参数 (a, v, t, z 组水平)
        core_params = ["a", "v(0)", "v(1)", "t", "z_trans"]
        for param in core_params:
            if param in stats.index:
                r_val = stats.loc[param, rhat_col]
                status = "✅" if r_val < 1.01 else "⚠️"
                print(f"    Rhat({param}) = {r_val:.4f} {status}")
                if r_val >= 1.01:
                    all_rhat_warnings.append({
                        "group": fname, "parameter": param, "Rhat": r_val
                    })

    # ---- v2 新增: 计算 DIC ----
    try:
        dic_val = model.dic
        print(f"  DIC = {dic_val:.2f}")
        # 保存 DIC
        dic_info = dic_val if isinstance(dic_val, dict) else {"DIC": dic_val}
    except Exception as e:
        dic_info = {"DIC": np.nan, "error": str(e)}
        print(f"  ⚠️ DIC 计算失败: {e}")

    # ---- v2 新增: 参数边界检查 ----
    try:
        boundary_params = ["t", "z_trans"]
        for param in boundary_params:
            if param in stats.index:
                q025 = stats.loc[param, "2.5q"]
                q975 = stats.loc[param, "97.5q"]
                mean_val = stats.loc[param, "mean"]
                # t 不应接近 0, z_trans 不应接近极端值
                if param == "t" and q025 < 0.01:
                    all_boundary_warnings.append({
                        "group": fname, "parameter": param,
                        "issue": f"t_2.5q={q025:.4f} 接近 0",
                        "mean": mean_val
                    })
                    print(f"  ⚠️ 边界警告: t 的 2.5% 分位数 = {q025:.4f}，接近下限")
                if param == "z_trans" and (q025 < -4 or q975 > 4):
                    all_boundary_warnings.append({
                        "group": fname, "parameter": param,
                        "issue": f"z_trans [2.5q={q025:.2f}, 97.5q={q975:.2f}] 趋近极端值",
                        "mean": mean_val
                    })
                    print(f"  ⚠️ 边界警告: z_trans 95%CI [{q025:.2f}, {q975:.2f}] 趋近极端值")
    except Exception as e:
        print(f"  ⚠️ 边界检查失败: {e}")

    # 保存完整迹线
    try:
        traces_raw = model.get_traces()
        traces_simple = {}
        for key, val in traces_raw.items():
            try:
                arr = np.asarray(val, dtype=float).flatten()
                if len(arr) > 0:
                    traces_simple[key] = arr
            except Exception:
                continue

        if traces_simple:
            trace_path = OUT_DIR / f"{fname}_traces.pkl"
            with open(trace_path, "wb") as f:
                pickle.dump(traces_simple, f, protocol=pickle.HIGHEST_PROTOCOL)
            print(f"  迹线 (pickle) -> {trace_path}")

            npz_path = OUT_DIR / f"{fname}_traces.npz"
            np.savez_compressed(npz_path, **traces_simple)
            print(f"  迹线 (npz) -> {npz_path}")
            print(f"  参数数量: {len(traces_simple)}")
        else:
            print("  ⚠️ get_traces() 返回空，仅保存了统计文件")
    except Exception as e:
        print(f"  ⚠️ 迹线保存失败: {e}")
        print(f"  统计文件仍然可用")

    # 保存模型引用
    all_models[fname] = {
        "model": model,
        "stats": stats,
        "dic": dic_info,
        "n_subj": n_subj,
        "n_trials": len(df),
        "accuracy": acc
    }

    # v2 改进: 保留 MCMC 数据库文件（不再删除）
    print(f"  数据库文件保留: {db_name}")

print(f"\n{'=' * 50}")
print("✅ 所有拟合完成!")
print(f"结果保存在: {OUT_DIR}")
print(f"MCMC 数据库: {DB_DIR}")

# ---- v2 新增: 输出收敛与边界警告汇总 ----
if all_rhat_warnings:
    print(f"\n⚠️ Rhat 收敛警告 ({len(all_rhat_warnings)} 项):")
    for w in all_rhat_warnings:
        print(f"  {w['group']}: {w['parameter']} Rhat={w['Rhat']:.4f}")
else:
    print("\n✅ 所有参数的 Rhat 值正常")

if all_boundary_warnings:
    print(f"\n⚠️ 参数边界警告 ({len(all_boundary_warnings)} 项):")
    for w in all_boundary_warnings:
        print(f"  {w['group']}: {w['parameter']} - {w['issue']}")
else:
    print("\n✅ 所有参数未触及不合理边界")

---
## Step 3: 后验预测检查 (PPC) — v2 新增

使用 HDDM 内置工具生成后验预测数据，比较观测 RT 分布与模型预测分布。

如果 PPC 显示模型预测与观测数据一致，说明模型能够有效复现行为特征。

In [ ]:
# ============================================================
# Step 3: 后验预测检查 (PPC) — v2 新增
# ============================================================

PPC_DIR = BASE_DIR / "3_Figures" / "HDDM_Results" / "PPC"
PPC_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Step 3: 后验预测检查 (PPC)")
print("=" * 60)

ppc_summary = []

for csv_path in csv_files:
    fname = csv_path.stem
    print(f"\n{'=' * 50}")
    print(f"PPC: {fname}")
    print(f"{'=' * 50}")

    if fname not in all_models:
        print(f"  ⚠️ 未找到 {fname} 的模型，跳过")
        continue

    model = all_models[fname]["model"]
    df = pd.read_csv(csv_path)

    try:
        # 生成后验预测数据
        # post_pred_gen 生成后验预测的 RT 和 response
        print("  生成后验预测数据 (samples=50)...")
        ppc_data = hddm.utils.post_pred_gen(model, samples=50)

        # 计算后验预测统计量
        try:
            ppc_stats = hddm.utils.post_pred_stats(model, samples=50)
            print(f"  后验预测统计:")
            if isinstance(ppc_stats, dict):
                for k, v in ppc_stats.items():
                    if isinstance(v, (int, float)):
                        print(f"    {k}: {v:.4f}")
                    else:
                        print(f"    {k}: {v}")
            else:
                print(f"    {ppc_stats}")
        except Exception as e:
            print(f"  ⚠️ post_pred_stats 失败: {e}")

        # 可视化: 观测 vs 预测 RT 分布
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # 观测数据 RT 分布
        for idx, (identity_val, label, color) in enumerate([
            (1, "Self", "#2196F3"), (0, "Stranger", "#FF9800")
        ]):
            obs_rt = df[(df["identity"] == identity_val) & (df["response"] == 1)]["rt"]
            if len(obs_rt) > 0:
                axes[0].hist(obs_rt, bins=30, alpha=0.5, color=color, label=f"{label} (n={len(obs_rt)})", density=True)
        axes[0].set_xlabel("RT (s)"); axes[0].set_ylabel("Density")
        axes[0].set_title(f"Observed RT Distribution\n{fname}")
        axes[0].legend(fontsize=8)

        # 尝试绘制后验预测 RT
        if isinstance(ppc_data, dict) and len(ppc_data) > 0:
            # ppc_data 可能包含模拟的 RT 数据
            try:
                if "rt" in ppc_data:
                    pred_rt = ppc_data["rt"]
                    if isinstance(pred_rt, np.ndarray):
                        pred_rt = pred_rt.flatten()
                    axes[1].hist(pred_rt, bins=30, alpha=0.6, color="#4CAF50", density=True, label="Predicted")
                elif isinstance(ppc_data, np.ndarray):
                    axes[1].hist(ppc_data.flatten(), bins=30, alpha=0.6, color="#4CAF50", density=True, label="Predicted")
            except Exception:
                pass

        # 叠加观测分布作为参考
        all_obs_rt = df[df["response"] == 1]["rt"]
        if len(all_obs_rt) > 0:
            axes[1].hist(all_obs_rt, bins=30, alpha=0.3, color="gray", density=True, label="Observed")

        axes[1].set_xlabel("RT (s)")
        axes[1].set_title(f"PPC: Observed vs Predicted RT\n{fname}")
        axes[1].legend(fontsize=8)

        plt.tight_layout()
        ppc_fig_path = PPC_DIR / f"{fname}_ppc.png"
        plt.savefig(ppc_fig_path, dpi=150, bbox_inches="tight")
        plt.show()

        # 保存 PPC 数据
        try:
            ppc_data_path = OUT_DIR / f"{fname}_ppc_data.pkl"
            with open(ppc_data_path, "wb") as f:
                pickle.dump(ppc_data, f)
        except Exception:
            pass

        ppc_summary.append({"group": fname, "ppc": "completed"})

    except Exception as e:
        print(f"  ❌ PPC 失败: {e}")
        ppc_summary.append({"group": fname, "ppc": f"error: {e}"})

print(f"\n{'=' * 50}")
print("✅ PPC 完成!")
print(f"结果保存在: {PPC_DIR}")

---
## Step 4: 参数提取（v2 改进）

提取 DDM 参数的组水平与个体水平后验估计。
**v2 改进**: SPE_v 从后验样本直接计算（而非仅用点估计相减），正确反映不确定性。

In [ ]:
# ============================================================
# Step 4: 参数提取（v2 改进: 组水平 + 个体水平）
# ============================================================

print("=" * 60)
print("Step 4: 提取 HDDM 参数后验分布")
print("=" * 60)

stats_files = sorted(OUT_DIR.glob("*_stats.csv"))

if not stats_files:
    print("未找到 stats 文件！请先运行 Step 2。")
else:
    all_group_params = []  # 组水平参数
    all_subject_params = []  # 个体水平参数

    for stats_path in stats_files:
        fname = stats_path.stem.replace("_stats", "")
        print(f"\n处理: {fname}")

        match = re.search(r"group(\d+)_P(\d+)_T(\d+)_W(\d+)", fname)
        if not match:
            print(f"  ⚠️ 无法解析文件名: {fname}，跳过")
            continue
        group_id = int(match.group(1))
        P_val = int(match.group(2))
        T_val = int(match.group(3))
        W_val = int(match.group(4))

        stats = pd.read_csv(stats_path, index_col=0)

        # ====== 组水平参数提取 ======
        row_group = {
            "group_id": group_id, "P": P_val, "T_ms": T_val, "W_ms": W_val
        }

        for stat_key, label in [
            ("v(0)", "v_stranger"), ("v(1)", "v_self"),
            ("a", "a"), ("t", "t"), ("z_trans", "z")
        ]:
            if stat_key in stats.index:
                row_group[f"{label}_mean"] = float(stats.loc[stat_key, "mean"])
                row_group[f"{label}_std"] = float(stats.loc[stat_key, "std"])
                row_group[f"{label}_q025"] = float(stats.loc[stat_key, "2.5q"])
                row_group[f"{label}_q975"] = float(stats.loc[stat_key, "97.5q"])

        # v2 改进: SPE_v 从后验样本计算
        trace_path = OUT_DIR / f"{fname}_traces.npz"
        if trace_path.exists():
            traces = np.load(trace_path, allow_pickle=True)
            if "v(1)" in traces and "v(0)" in traces:
                v_self_samples = traces["v(1)"]
                v_stranger_samples = traces["v(0)"]
                spe_v_samples = v_self_samples - v_stranger_samples
                row_group["SPE_v_mean"] = float(np.mean(spe_v_samples))
                row_group["SPE_v_std"] = float(np.std(spe_v_samples))
                row_group["SPE_v_q025"] = float(np.percentile(spe_v_samples, 2.5))
                row_group["SPE_v_q975"] = float(np.percentile(spe_v_samples, 97.5))
                print(f"  SPE_v = {row_group['SPE_v_mean']:.3f} [95%CI: {row_group['SPE_v_q025']:.3f}, {row_group['SPE_v_q975']:.3f}]")
        else:
            # 回退: 使用点估计相减
            if "v_self_mean" in row_group and "v_stranger_mean" in row_group:
                row_group["SPE_v_mean"] = row_group["v_self_mean"] - row_group["v_stranger_mean"]
                print(f"  ⚠️ 迹线不可用，SPE_v 使用点估计: {row_group['SPE_v_mean']:.3f}")

        # 添加 DIC
        if fname in all_models:
            dic_info = all_models[fname]["dic"]
            if isinstance(dic_info, dict):
                for dik, div in dic_info.items():
                    if isinstance(div, (int, float)):
                        row_group[f"DIC_{dik}"] = div
            elif isinstance(dic_info, (int, float)):
                row_group["DIC"] = dic_info

        all_group_params.append(row_group)

        # ====== v2 新增: 个体水平参数提取 ======
        n_subjects = int(row_group.get("n_subjects", 
            sum(1 for c in stats.index if c.startswith("a_subj."))))

        for subj_idx in range(n_subjects):
            row_subj = {
                "group_id": group_id, "P": P_val, "T_ms": T_val, "W_ms": W_val,
                "subj_idx": subj_idx
            }
            for param, prefix in [("a", "a_subj"), ("t", "t_subj"), ("z", "z_subj_trans")]:
                key = f"{prefix}.{subj_idx}"
                if key in stats.index:
                    row_subj[f"{param}_mean"] = float(stats.loc[key, "mean"])
                    row_subj[f"{param}_std"] = float(stats.loc[key, "std"])
                    row_subj[f"{param}_q025"] = float(stats.loc[key, "2.5q"])
                    row_subj[f"{param}_q975"] = float(stats.loc[key, "97.5q"])
            # 提取 v_self 和 v_stranger 的个体估计
            for identity_val, label in [(0, "v_stranger"), (1, "v_self")]:
                key = f"v_subj({identity_val}).{subj_idx}"
                if key in stats.index:
                    row_subj[f"{label}_mean"] = float(stats.loc[key, "mean"])
                    row_subj[f"{label}_std"] = float(stats.loc[key, "std"])
            all_subject_params.append(row_subj)

    # ====== 保存 ======
    df_group = pd.DataFrame(all_group_params).sort_values("group_id")
    df_subject = pd.DataFrame(all_subject_params).sort_values(["group_id", "subj_idx"])

    df_group.to_csv(OUT_DIR / "all_groups_ddm_params.csv", index=False)
    df_subject.to_csv(OUT_DIR / "all_subjects_ddm_params.csv", index=False)

    # 显示组水平汇总
    disp_cols = ["group_id", "P", "T_ms", "W_ms",
                 "v_self_mean", "v_stranger_mean", "SPE_v_mean",
                 "a_mean", "t_mean"]
    avail = [c for c in disp_cols if c in df_group.columns]
    print("\n" + "=" * 60)
    print("DDM 参数汇总 (组水平)")
    print("=" * 60)
    print(df_group[avail].round(4).to_string(index=False))

    # 显示 DIC（如有）
    dic_cols = [c for c in df_group.columns if c.startswith("DIC")]
    if dic_cols:
        print("\n" + "=" * 60)
        print("DIC 拟合优度")
        print("=" * 60)
        print(df_group[["group_id"] + dic_cols].round(2).to_string(index=False))

    print(f"\n✅ 组水平参数: {OUT_DIR / 'all_groups_ddm_params.csv'}")
    print(f"✅ 个体水平参数: {OUT_DIR / 'all_subjects_ddm_params.csv'}")

---
## Step 5: 综合可视化（v2 增强）

包含：收敛诊断图（迹线图 + 后验密度图）+ DDM 参数组间比较图。

In [ ]:
# ============================================================
# Step 5: 综合可视化（v2 增强）
# ============================================================

plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

FIG_DIR = BASE_DIR / "3_Figures" / "HDDM_Results"
TRACE_DIR = FIG_DIR / "Traces"
TRACE_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Step 5: 综合可视化")
print("=" * 60)

# ================================================================
# Part A: 收敛诊断图（迹线图 + 后验密度图）
# ================================================================
print("\n--- Part A: 收敛诊断图 ---")

for csv_path in csv_files:
    fname = csv_path.stem
    trace_path = OUT_DIR / f"{fname}_traces.npz"

    if not trace_path.exists():
        print(f"  跳过 {fname}: 迹线文件不存在")
        continue

    traces = np.load(trace_path, allow_pickle=True)
    # 选择核心参数（组水平）: a, v(0), v(1), t, z_trans
    core_keys = ["a", "v(0)", "v(1)", "t", "z_trans"]
    available_keys = [k for k in core_keys if k in traces]

    if not available_keys:
        print(f"  跳过 {fname}: 无核心参数")
        continue

    n_params = len(available_keys)
    fig, axes = plt.subplots(n_params, 2, figsize=(12, 3 * n_params))
    if n_params == 1:
        axes = np.array([axes])

    for i, key in enumerate(available_keys):
        samples = traces[key].flatten()

        # 迹线图 (左列)
        axes[i, 0].plot(samples, color="#2196F3", alpha=0.7, linewidth=0.5)
        axes[i, 0].set_ylabel(key, fontsize=11)
        axes[i, 0].set_title(f"Trace: {key}" if i == 0 else f"Trace: {key}")
        axes[i, 0].axhline(y=np.mean(samples), color="red", linestyle="--", alpha=0.6, linewidth=1)

        # 后验密度图 (右列)
        axes[i, 1].hist(samples, bins=50, color="#4CAF50", alpha=0.7, density=True, edgecolor="white")
        axes[i, 1].axvline(x=np.mean(samples), color="red", linestyle="--", alpha=0.8)
        axes[i, 1].axvline(x=np.percentile(samples, 2.5), color="gray", linestyle=":", alpha=0.6)
        axes[i, 1].axvline(x=np.percentile(samples, 97.5), color="gray", linestyle=":", alpha=0.6)
        axes[i, 1].set_title(f"Posterior: {key}")
        axes[i, 1].set_ylabel("Density")

    plt.suptitle(f"Convergence Diagnostics: {fname}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    trace_fig_path = TRACE_DIR / f"{fname}_diagnostics.png"
    plt.savefig(trace_fig_path, dpi=200, bbox_inches="tight")
    plt.show()

    print(f"  ✅ {fname} 诊断图 -> {trace_fig_path}")

print("\n--- Part B: 参数组间比较图 ---")

# ================================================================
# Part B: DDM 参数组间比较图（v2 改进: 正确误差条）
# ================================================================

# 重新加载参数表（如果之前步骤未执行）
param_path = OUT_DIR / "all_groups_ddm_params.csv"
if param_path.exists():
    df_params = pd.read_csv(param_path).sort_values("group_id")
else:
    print("  参数表不存在，跳过 Part B")
    df_params = None

if df_params is not None and len(df_params) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # ---- (0,0): SPE_v 带正确 95% CI ----
    ax = axes[0, 0]
    if "SPE_v_mean" in df_params.columns:
        valid = df_params.dropna(subset=["SPE_v_mean"]).reset_index(drop=True)
        colors = plt.cm.viridis(np.linspace(0, 1, len(valid)))
        for i, (_, r) in enumerate(valid.iterrows()):
            # v2 修复: 使用 SPE_v 自身的 95% CI
            if "SPE_v_q025" in valid.columns and "SPE_v_q975" in valid.columns:
                lo = r["SPE_v_mean"] - r["SPE_v_q025"]
                hi = r["SPE_v_q975"] - r["SPE_v_mean"]
            else:
                lo = r["SPE_v_std"] if "SPE_v_std" in valid.columns else 0
                hi = lo
            ax.errorbar(i, r["SPE_v_mean"], yerr=[[lo], [hi]],
                        fmt="o", color=colors[i], capsize=5, markersize=8, markeredgecolor="black")
        ax.axhline(y=0, color="gray", linestyle="--", linewidth=1)
        ax.set_xticks(range(len(valid)))
        ax.set_xticklabels([f"G{r['group_id']:.0f}" for _, r in valid.iterrows()], rotation=45)
        ax.set_ylabel("SPE_v (Self - Stranger)")
        ax.set_title("Self-Prioritization Effect in Drift Rate")

    # ---- (0,1): v_self vs v_stranger ----
    ax = axes[0, 1]
    if "v_self_mean" in df_params.columns:
        valid = df_params.dropna(subset=["v_self_mean"]).reset_index(drop=True)
        x = np.arange(len(valid))
        w = 0.35
        # Self
        bar1 = ax.bar(x - w/2, valid["v_self_mean"], w, label="Self", color="#2196F3", alpha=0.85, edgecolor="white")
        # Stranger
        bar2 = ax.bar(x + w/2, valid["v_stranger_mean"], w, label="Stranger", color="#FF9800", alpha=0.85, edgecolor="white")
        # 误差线 (95% CI)
        for i, (_, r) in enumerate(valid.iterrows()):
            if "v_self_q025" in valid.columns:
                ax.errorbar(x[i] - w/2, r["v_self_mean"],
                            yerr=[[r["v_self_mean"] - r["v_self_q025"]], [r["v_self_q975"] - r["v_self_mean"]]],
                            fmt="none", color="black", capsize=2, linewidth=0.8)
            if "v_stranger_q025" in valid.columns:
                ax.errorbar(x[i] + w/2, r["v_stranger_mean"],
                            yerr=[[r["v_stranger_mean"] - r["v_stranger_q025"]], [r["v_stranger_q975"] - r["v_stranger_mean"]]],
                            fmt="none", color="black", capsize=2, linewidth=0.8)
        ax.axhline(y=0, color="gray", linestyle="--", linewidth=1)
        ax.set_xticks(x)
        ax.set_xticklabels([f"G{r['group_id']:.0f}" for _, r in valid.iterrows()], rotation=45)
        ax.set_ylabel("Drift Rate v")
        ax.set_title("Drift Rate by Condition (95% CI)")
        ax.legend()

    # ---- (1,0): 边界 a ----
    ax = axes[1, 0]
    if "a_mean" in df_params.columns:
        valid = df_params.dropna(subset=["a_mean"]).reset_index(drop=True)
        x = np.arange(len(valid))
        ax.bar(x, valid["a_mean"], color="#607D8B", alpha=0.85, edgecolor="white")
        if "a_q025" in valid.columns:
            ax.errorbar(x, valid["a_mean"],
                        yerr=[valid["a_mean"] - valid["a_q025"], valid["a_q975"] - valid["a_mean"]],
                        fmt="none", color="black", capsize=4, linewidth=1.2)
        ax.set_xticks(x)
        ax.set_xticklabels([f"G{r['group_id']:.0f}" for _, r in valid.iterrows()], rotation=45)
        ax.set_ylabel("Boundary a")
        ax.set_title("Decision Boundary (95% CI)")

    # ---- (1,1): 非决策时间 t ----
    ax = axes[1, 1]
    if "t_mean" in df_params.columns:
        valid = df_params.dropna(subset=["t_mean"]).reset_index(drop=True)
        x = np.arange(len(valid))
        ax.bar(x, valid["t_mean"], color="#E91E63", alpha=0.85, edgecolor="white")
        if "t_q025" in valid.columns:
            ax.errorbar(x, valid["t_mean"],
                        yerr=[valid["t_mean"] - valid["t_q025"], valid["t_q975"] - valid["t_mean"]],
                        fmt="none", color="black", capsize=4, linewidth=1.2)
        ax.set_xticks(x)
        ax.set_xticklabels([f"G{r['group_id']:.0f}" for _, r in valid.iterrows()], rotation=45)
        ax.set_ylabel("t (s)")
        ax.set_title("Nondecision Time (95% CI)")

    plt.suptitle("DDM Parameters by Experimental Group", fontsize=14, fontweight="bold")
    plt.tight_layout()
    fig_path = FIG_DIR / "ddm_params_by_group_v2.png"
    plt.savefig(fig_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"\n✅ 参数比较图 -> {fig_path}")
else:
    print("  无参数数据可绘图")

---
## 输出文件清单

| 文件 | 位置 | 内容 | v2 变更 |
|------|------|------|---------|
| `hddm_data_group*.csv` | `2_Data/Real_Data/HDDM_Ready/` | Step 1 预处理后的 HDDM 输入（仅有效试次） | 排除遗漏试次 |
| `omission_behavioral_summary.csv` | `2_Data/Real_Data/HDDM_Traces/` | 各组遗漏率行为指标 | **新增** |
| `*_stats.csv` | `2_Data/Real_Data/HDDM_Traces/` | Step 2 各参数后验摘要 (mean, std, 分位数, Rhat) | 含 Rhat |
| `*_traces.pkl` | `2_Data/Real_Data/HDDM_Traces/` | Step 2 完整 MCMC 采样迹线 (pickle) | |
| `*_traces.npz` | `2_Data/Real_Data/HDDM_Traces/` | Step 2 完整 MCMC 采样迹线 (numpy) | |
| `traces_*.db` | `2_Data/Real_Data/HDDM_Traces/mcmc_databases/` | MCMC 原始数据库（供诊断） | **保留** |
| `*_ppc_data.pkl` | `2_Data/Real_Data/HDDM_Traces/` | 后验预测数据 | **新增** |
| `all_groups_ddm_params.csv` | `2_Data/Real_Data/HDDM_Traces/` | 所有条件组水平汇总（含 SPE_v 95% CI, DIC） | 增强 |
| `all_subjects_ddm_params.csv` | `2_Data/Real_Data/HDDM_Traces/` | 所有被试个体水平参数 | **新增** |
| `*_diagnostics.png` | `3_Figures/HDDM_Results/Traces/` | 收敛诊断图（迹线 + 密度） | **新增** |
| `*_ppc.png` | `3_Figures/HDDM_Results/PPC/` | 后验预测检查图 | **新增** |
| `ddm_params_by_group_v2.png` | `3_Figures/HDDM_Results/` | 参数组间比较图（正确 95% CI） | 修复误差条 |

---
## v2.0 与 v1.0 关键差异总结

1. **遗漏试次处理**: v1 将遗漏试次作为错误反应纳入拟合 → v2 排除遗漏试次，仅对有效试次建模
2. **收敛诊断**: v1 无 → v2 包含 Rhat 检查、迹线图、后验密度图
3. **模型评估**: v1 无 PPC/DIC → v2 包含 PPC 和 DIC
4. **参数提取**: v1 仅组水平点估计 → v2 组水平 + 个体水平 + SPE_v 完整后验分布
5. **可视化**: v1 基础柱状图 → v2 含 95% CI 误差条、收敛诊断图、PPC 图

---
## 注意事项

1. **遗漏试次**: 已完全排除。遗漏率作为独立行为指标保存在 `omission_behavioral_summary.csv`
2. **数据路径**: 所有输出直接写入项目根目录的 `2_Data/` 和 `3_Figures/` 下
3. **重新拟合**: 如需重新拟合，直接重新运行 Step 1-5 即可（会覆盖已有文件）
4. **MCMC 链**: 当前使用默认链数。如需多链 Rhat，可在 Step 2 中设置 `db="txt"` 并运行多个独立链